# Notebook 04 — CMF-Static Unlearning · CIFAR-100 (Paper Protocol)

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Protocol & Overview:**
- Loads `theta_o` from NB1 (CIFAR-100) into a `ModelModule` (with CMF head).
- Runs `run_cmf_static()` (paper Algorithm 2: per-epoch recompute_cmf → freeze W → update encoder) for all 5 base methods:
  `grad_ascent_descent`, `random_label`, `salun`, `scrub`, `tarun`.
- Hyperparameters from Table 4 CIFAR-100 column.
- 5 representative CIFAR-100 forget classes `{0},{1},{2},{3},{5}` (§A.3).
- Saves self-describing checkpoints: `{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}.pt`.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path containing theta_o checkpoint from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-nb1-c100'

SPLIT_CFG = os.path.join(CKPT_DATASET_DIR, 'split_config_cifar100.json')
if os.path.exists(SPLIT_CFG):
    with open(SPLIT_CFG) as f: NB1_CFG = json.load(f)
    DATASET     = NB1_CFG.get('dataset',     'cifar100')
    NUM_CLASSES = NB1_CFG.get('num_classes', 100)
    TEST_MODE   = NB1_CFG.get('test_mode',   False)
else:
    DATASET     = 'cifar100'
    NUM_CLASSES = 100
    TEST_MODE   = False

ARCH = 'resnet18'
STAGE = '4a'

# Paper §A.3: CIFAR-100 single-class forget — 5 representative classes
FORGET_CLASSES = [0, 1, 2, 3, 5]
SEEDS          = [0]
THETA_O_SEED   = 0
BASE_METHODS   = ['grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun']
MEAN_SOURCES   = ['train']

# Hyperparameters — CIFAR-100 column of Table 4
CMF_EPOCHS_BY_METHOD = {
    'random_label':        1 if TEST_MODE else 4,   # Table 4 line 2393
    'salun':               1 if TEST_MODE else 4,   # Table 4 line 2402
    'grad_ascent_descent': 1 if TEST_MODE else 3,   # Table 4 line 2411
    'scrub':               1 if TEST_MODE else 3,   # Table 4 line 2420
    'tarun':               1 if TEST_MODE else 3,   # Table 4 line 2429
}
CMF_LR = {
    'random_label':        2e-3,   # Table 4 line 2393
    'salun':               2e-3,   # Table 4 line 2402
    'grad_ascent_descent': 1e-4,   # Table 4 line 2411
    'scrub':               5e-3,   # Table 4 line 2420
    'tarun':               5e-5,   # Table 4 line 2429
}
CMF_BATCH = {
    'scrub': 64,   # Table 4 line 2420
}
CMF_BATCH_DEFAULT = 128

CKPT_ROOT = '/kaggle/working/checkpoints/cmf_cifar100_paper'
os.makedirs(f'{CKPT_ROOT}/cmf_static', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'STAGE={STAGE}  DATASET={DATASET}  ARCH={ARCH}  device={device}')
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}')

In [ ]:
import torchvision, torchvision.transforms as transforms

# CIFAR-100 standard normalization statistics
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

full_train      = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=False, transform=transform_test)
test_set        = torchvision.datasets.CIFAR100('/kaggle/working/data', train=False,
                                                download=True, transform=transform_test)

test_targets = torch.tensor(test_set.targets)
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
print(f'Train size: {len(full_train)}  Test size: {len(test_set)}')
print(f'Train samples per class: {len(TRAIN_CLASS_IDX[0])} (expected 500 for CIFAR-100)')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.CMF import run_cmf_static

def build_cmf_model(args):
    model = ModelModule(args).to(device)
    assert hasattr(model, 'CMFweights'), 'Model must have CMFweights attribute!'
    return model

def make_cmf_args(base_method, lr, epochs, mean_source, forget_class,
                  forget_train_idx, retain_train_idx, seed=0):
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        # SVD params — CIFAR-100: alpha_r=1000, alpha_f=30, samples=990 (Table 4 line 2380)
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=990, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        # UNSIR: impair_lr=5e-5 (Table 4 line 2429), same as main lr for CIFAR-100
        tarun_impair_lr=5e-5, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model._preprocess_feats_for_cmf(f)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=50):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc

def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)
    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    return ((ret_pred == yte_r).float().mean().item() * 100,
            (fgt_pred == yte_f).float().mean().item() * 100)
def eval_cmf_three_metrics(model,
                           test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt   = run_probe_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                      test_retain_ldr, test_forget_ldr)
    ncc_ret, ncc_fgt = run_ncc_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                    test_retain_ldr, test_forget_ldr)
    return {'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
            'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt}

print('Helpers ready.')

In [ ]:
# ── Load theta_o checkpoint from Notebook 1 (CIFAR-100) ─────────────────────
theta_o_path = os.path.join(CKPT_DATASET_DIR, 'original_resnet18_cifar100.pt')
assert os.path.exists(theta_o_path), (
    f'Missing theta_o: {theta_o_path}\n'
    f'Run Notebook 1 (CIFAR-100) first and set CKPT_DATASET_DIR correctly.'
)
ck_o = torch.load(theta_o_path, map_location=device)
theta_o_state = ck_o.get('model_state_dict', ck_o)
print(f'Loaded theta_o from {theta_o_path}')

results_4a = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]      # ~500 samples
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]                                                       # ~49 500 samples

        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_forget_idx = TEST_CLASS_IDX[forget_class]         # 100 test samples
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]                                                        # 9 900 test samples
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_loader_full = torch.utils.data.DataLoader(
            test_set, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                lr    = CMF_LR.get(base_method, 1e-3)
                batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                tag   = f'{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}'
                ckpt_path = f'{CKPT_ROOT}/cmf_static/{tag}.pt'

                if os.path.exists(ckpt_path):
                    print(f'[{tag}] exists — loading cached result.')
                    ck = torch.load(ckpt_path, map_location=device)
                    results_4a.append(ck['metrics'])
                    continue

                print(f'\n[{tag}] Running cmf_static ({base_method}) for {cmf_epochs} epochs...')
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                args = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                     forget_class, forget_train_idx, retain_train_idx,
                                     seed=seed)
                args.batch_size = batch
                if base_method == 'scrub':
                    args.scrub_del_bsz  = 64
                    args.scrub_sgda_bsz = 64

                model = build_cmf_model(args)
                model.encoder.load_state_dict(theta_o_state, strict=False)

                mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                t0 = time.time()
                try:
                    model = run_cmf_static(
                        base_method=base_method,
                        args=args,
                        model=model,
                        device=device,
                        retain_loader=retain_loader,
                        forget_loader=forget_loader,
                        train_loader=mean_loader,
                        test_loader=test_loader_full,
                        epochs=cmf_epochs,
                        test_forget_loader=test_forget_ldr,
                        train_dataset=full_train,
                        val_index=retain_train_idx,
                    )
                except Exception as e:
                    print(f'ERROR running {base_method}: {e}')
                    traceback.print_exc()
                    continue
                wall_min = (time.time() - t0) / 60

                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                metrics = eval_cmf_three_metrics(
                    model,
                    test_retain_ldr, test_forget_ldr,
                    train_retain_eval_ldr, train_forget_eval_ldr
                )
                metrics.update({
                    'method': base_method, 'mean_source': mean_source,
                    'forget_class': forget_class, 'seed': seed, 'stage': STAGE,
                    'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                    'wall_clock_minutes': wall_min,
                    'n_forget_train': len(forget_train_idx),
                    'n_retain_train': len(retain_train_idx),
                    'protocol': 'whole_class_single',
                })

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': {
                        'base_method': base_method, 'mean_source': mean_source,
                        'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                        'forget_class': forget_class, 'seed': seed,
                        'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                        'protocol': 'whole_class_single',
                        'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                        'paper': 'arXiv:2604.08271v1 Table 4 CIFAR-100',
                    },
                    'seed': seed, 'metrics': metrics,
                }, ckpt_path)
                print(f'  Saved {ckpt_path}')
                print(f'  out  R={metrics["output_retain_acc"]:.2f}%  F={metrics["output_forget_acc"]:.2f}%')
                print(f'  probe R={metrics["probe_retain_acc"]:.2f}%  F={metrics["probe_forget_acc"]:.2f}%')
                print(f'  ncc  R={metrics["ncc_retain_acc"]:.2f}%  F={metrics["ncc_forget_acc"]:.2f}%')
                results_4a.append(metrics)

df_4a = pd.DataFrame(results_4a)
csv_path = f'{CKPT_ROOT}/results_4a_cmf_static_cifar100.csv'
df_4a.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')
if not df_4a.empty:
    metric_cols = ['output_retain_acc','output_forget_acc',
                   'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']
    print('\n=== Summary ===')
    print(df_4a.groupby('method')[metric_cols].mean().round(2).to_string())